# LangGraph 활용 Agent 구축

이번 튜토리얼에서는 웹 검색 도구를 활용하여 챗봇에 웹 검색 기능을 수행하는 Agent를 구축합니다. LLM에 도구를 바인딩하면 LLM이 입력된 요청을 분석하여 필요한 경우 자동으로 웹 검색 도구(Tool)를 호출할 수 있습니다.

Agent는 단순한 챗봇과 달리, 주어진 질문에 대해 스스로 판단하여 도구를 사용할지 여부를 결정합니다. 예를 들어 "대한민국의 수도는?"과 같은 질문에는 LLM이 직접 답변하지만, "오늘 날씨는?"과 같은 최신 정보가 필요한 질문에는 웹 검색 도구를 호출하여 정보를 수집한 후 답변을 생성합니다.

이 튜토리얼에서는 조건부 엣지를 통해 도구 호출 여부에 따라 다른 노드로 라우팅하는 방법도 함께 학습합니다.

> 참고 문서: [LangGraph Agents](https://langchain-ai.github.io/langgraph/concepts/agentic_concepts/)

## 환경 설정

튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로깅 기능을 활성화하여 LangSmith에서 실행 추적을 확인할 수 있도록 합니다.

LangSmith 추적을 활성화하면 에이전트의 추론 과정, 도구 호출, 응답 생성 등을 시각적으로 디버깅할 수 있어 개발에 큰 도움이 됩니다.

아래 코드는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)
# 추적을 위한 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

LangChain/LangSmith API Key가 설정되지 않았습니다. 참고: https://wikidocs.net/250954


---

## 도구(Tool) 사용하기

챗봇이 학습 데이터만으로 답변할 수 없는 질문을 처리하기 위해 웹 검색 도구를 통합합니다. 도구를 사용하면 LLM이 실시간 정보를 검색하여 더 정확하고 최신의 응답을 제공할 수 있습니다. LangGraph에서는 도구를 LLM에 바인딩하여 에이전트가 필요에 따라 자동으로 도구를 호출하도록 구성할 수 있습니다.

> 참고 문서: [LangGraph Tools](https://langchain-ai.github.io/langgraph/concepts/agentic_concepts/#tool-calling-agent)

### 검색 API 도구

Tavily 검색 API를 활용하여 검색 기능을 구현하는 도구입니다. Tavily는 포괄적이고 정확하며 신뢰할 수 있는 결과에 최적화된 검색 엔진으로, 현재 이벤트에 대한 질문에 답변할 때 특히 유용합니다.

Tavily API 키를 발급받아 환경변수에 설정해야 합니다.

- API 키 발급 주소: https://app.tavily.com/

발급한 API 키를 `.env` 파일에 아래와 같이 설정합니다.

```
TAVILY_API_KEY=tvly-abcdefghijklmnopqrstuvwxyz
```

### 검색 도구 생성 및 테스트

웹 검색 도구인 `TavilySearch`를 생성합니다. `langchain_teddynote` 패키지에서 제공하는 `TavilySearch`는 Tavily API를 활용하여 웹 검색을 수행하고 결과를 반환합니다. `max_results` 파라미터를 통해 반환받을 검색 결과의 최대 개수를 설정할 수 있습니다.

아래 코드에서는 검색 도구를 생성하고 테스트 검색을 수행합니다.

In [2]:
from langchain_teddynote.tools.tavily import TavilySearch

# 검색 도구 생성 (최대 3개 결과 반환)
tool = TavilySearch(max_results=3)

# 도구 목록에 추가
tools = [tool]

# 도구 실행 테스트
print(tool.invoke("테디노트 랭체인 튜토리얼"))

[{'url': 'https://www.youtube.com/watch?v=mVu6Wj8Z7C0', 'title': '랭체인 한국어 튜토리얼     업데이트 소식   처음 사용자를 위한 친절한 ...', 'content': '#랭체인 한국어 튜토리얼🇰🇷 업데이트 소식🔥 처음 사용자를 위한 친절한 환경설치(Windows, Mac)\n테디노트 TeddyNote\n50200 subscribers\n339 likes\n20368 views\n19 Jun 2024\n📝 환경설정(Windows)\nhttps://teddynote.com/10-RAG%EB%B9%84%EB%B2%95%EB%85%B8%ED%8A%B8/%ED%99%98%EA%B2%BD%20%EC%84%A4%EC%A0%95%20(Windows)/\n\n📝 환경설정(Mac)\nhttps://teddynote.com/10-RAG%EB%B9%84%EB%B2%95%EB%85%B8%ED%8A%B8/%ED%99%98%EA%B2%BD%20%EC%84%A4%EC%A0%95%20(Mac)/\n\n📍[패스트캠퍼스] "테디노트의 RAG 비법노트" 강의\n링크: https://bit.ly/4e1h8zO\n\n🤖 디스코드 채널\nhttps://discord.gg/q3RvQZ5CfK\n\n📘 랭체인 튜토리얼 무료 전자책(wikidocs)\nhttps://wikidocs.net/book/14314\n\n✅ 랭체인 한국어 튜토리얼 코드저장소(GitHub)\nhttps://github.com/teddylee777/langchain-kr\n\n✅ 줄거리\n00:00 랭체인 한국어 튜토리얼 공지사항\n01:59 langchain-teddynote 패키지\n08:25 감사인사\n09:15 Windows 환경설치\n21:48 Mac 환경설치\n\n#rag #langchain\n---\n📍 "테디노트의 RAG 비법노트" 랭체인 강의: https://fastcampus.co.kr/data_online_teddy\n📘 랭체인 한국어 튜토리얼(무료 전자책)

### State 정의

검색 결과는 챗봇이 질문에 답할 수 있도록 사용할 수 있는 페이지 요약입니다. 이번에는 LLM에 `bind_tools`를 추가하여 **LLM + 도구** 조합을 구성합니다. `bind_tools` 메서드는 LLM이 어떤 도구를 사용할 수 있는지 알려주고, 필요한 경우 도구 호출을 생성할 수 있게 합니다.

먼저 그래프에서 사용할 State를 정의합니다. `messages` 필드에 `add_messages` 리듀서를 적용하면, 새 메시지가 기존 리스트를 덮어쓰지 않고 자동으로 추가됩니다.

아래 코드에서는 State를 정의합니다.

In [3]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages


# State 정의
class State(TypedDict):
    """에이전트의 상태를 정의하는 타입

    messages: 대화 메시지 리스트
    - add_messages 리듀서를 통해 새 메시지가 자동으로 추가됩니다
    """

    # list 타입에 add_messages 적용(list 에 message 추가)
    messages: Annotated[list, add_messages]

### LLM 정의 및 도구 바인딩

LLM을 정의하고 앞서 생성한 도구를 바인딩합니다. `bind_tools` 메서드를 사용하면 LLM이 입력을 분석하여 필요한 경우 적절한 도구를 호출할 수 있게 됩니다. 도구가 바인딩된 LLM은 사용자의 질문을 받았을 때 직접 답변할 수 있으면 바로 응답하고, 외부 정보가 필요하면 도구 호출을 생성합니다.

아래 코드에서는 `init_chat_model`을 사용하여 모델을 초기화하고 검색 도구를 바인딩합니다.

In [4]:
from langchain.chat_models import init_chat_model

# LLM 초기화
# OpenAI 키 사용 시 gpt-5.2, gpt-4.1-mini 등으로 변경 가능
llm = init_chat_model("claude-sonnet-4-5")

# LLM 에 도구 바인딩
llm_with_tools = llm.bind_tools(tools)

### 노드 함수 정의

챗봇 노드 함수는 현재 상태(State)를 입력으로 받아 LLM을 호출하고, 응답 메시지를 포함한 딕셔너리를 반환합니다. `llm_with_tools`는 도구가 바인딩된 LLM이므로, 필요한 경우 자동으로 도구 호출(tool_calls)을 생성합니다.

반환된 메시지는 `add_messages` 리듀서에 의해 기존 메시지 목록에 자동으로 추가됩니다. 이 과정에서 개발자가 별도로 메시지를 관리할 필요가 없습니다.

아래 코드에서는 챗봇 노드 함수를 정의합니다.

In [5]:
# 챗봇 노드 함수 정의
def chatbot(state: State):
    """챗봇 노드 함수

    현재 상태의 메시지를 받아 도구가 바인딩된 LLM을 호출하고,
    응답을 메시지 리스트로 반환합니다.
    """
    # LLM 호출 및 응답 생성
    answer = llm_with_tools.invoke(state["messages"])
    # 메시지 목록 반환 (자동으로 add_messages 적용)
    return {"messages": [answer]}

### 그래프 생성 및 노드 추가

`StateGraph`를 생성하고 정의한 챗봇 노드를 추가합니다. `StateGraph`는 정의한 `State` 타입을 기반으로 상태 기반 워크플로우를 구성합니다. `add_node()` 메서드를 사용하여 "chatbot"이라는 이름으로 노드를 추가합니다.

이 노드는 사용자 입력을 받아 LLM 응답을 생성하는 역할을 합니다. 노드 이름은 그래프 내에서 고유해야 하며, 엣지를 정의할 때 이 이름을 사용하여 노드 간의 연결을 설정합니다.

아래 코드에서는 StateGraph를 생성하고 챗봇 노드를 추가합니다.

In [6]:
from langgraph.graph import StateGraph

# 상태 그래프 초기화
graph_builder = StateGraph(State)

# 챗봇 노드 추가
graph_builder.add_node("chatbot", chatbot)

---

## 도구 노드(Tool Node)

도구 노드는 LLM이 생성한 도구 호출 요청을 실제로 실행하는 역할을 합니다. 챗봇 노드가 도구 호출이 필요하다고 판단하면 `tool_calls`가 포함된 `AIMessage`를 반환하고, 도구 노드가 이를 받아 실제 도구를 실행한 후 결과를 `ToolMessage`로 반환합니다.

이번 섹션에서는 `BasicToolNode` 클래스를 직접 구현하여 도구 호출의 내부 동작을 이해합니다. LangGraph에서는 이 기능을 제공하는 pre-built [ToolNode](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.tool_node.ToolNode)도 제공하고 있습니다.

### BasicToolNode 구현

가장 최근의 메시지를 확인하고 메시지에 `tool_calls`가 포함되어 있으면 도구를 호출하는 `BasicToolNode`를 구현합니다. 이 클래스는 도구 이름을 키로, 도구 객체를 값으로 하는 딕셔너리를 관리하며, AIMessage의 `tool_calls`에 지정된 도구를 순서대로 실행합니다.

도구 실행 결과는 `ToolMessage` 형태로 반환되며, 각 결과에는 도구 이름과 호출 ID가 포함됩니다. 이를 통해 LLM이 어떤 도구 호출에 대한 결과인지 정확하게 매칭할 수 있습니다.

아래 코드에서는 `BasicToolNode` 클래스를 구현하고 그래프에 도구 노드를 추가합니다.

In [7]:
import json
from langchain_core.messages import ToolMessage


class BasicToolNode:
    """마지막 AIMessage에서 요청된 도구를 실행하는 노드"""

    def __init__(self, tools: list) -> None:
        # 도구 이름을 키로, 도구 객체를 값으로 하는 딕셔너리 생성
        self.tools_list = {tool.name: tool for tool in tools}

    def __call__(self, inputs: dict):
        # 메시지가 존재할 경우 가장 최근 메시지 1개 추출
        if messages := inputs.get("messages", []):
            message = messages[-1]
        else:
            raise ValueError("No message found in input")

        # 도구 호출 결과를 저장할 리스트
        outputs = []
        for tool_call in message.tool_calls:
            # 도구 호출 후 결과 저장
            tool_result = self.tools_list[tool_call["name"]].invoke(tool_call["args"])
            outputs.append(
                # 도구 호출 결과를 ToolMessage로 변환
                ToolMessage(
                    content=json.dumps(
                        tool_result, ensure_ascii=False
                    ),  # 도구 호출 결과를 문자열로 변환
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )

        return {"messages": outputs}


# 도구 노드 생성
tool_node = BasicToolNode(tools=[tool])

# 그래프에 도구 노드 추가
graph_builder.add_node("tools", tool_node)

---

## 조건부 엣지(Conditional Edge)

도구 노드가 추가되면 `conditional_edges`를 정의할 수 있습니다. 일반 엣지(Edge)는 한 노드에서 다음 노드로 제어 흐름을 고정적으로 라우팅하지만, 조건부 엣지(Conditional Edge)는 현재 그래프 상태에 따라 다른 노드로 동적으로 라우팅합니다.

조건부 엣지 함수는 현재 그래프 `state`를 받아 다음에 호출할 노드를 나타내는 **문자열 또는 문자열 목록**을 반환합니다. 도구 호출이 있으면 `"tools"` 노드로, 도구 호출이 없으면 `END`로 라우팅하여 그래프 실행을 종료합니다.

아래에서는 `route_tools`라는 라우터 함수를 정의하여 챗봇의 출력에서 `tool_calls`를 확인하고, 이 함수를 `add_conditional_edges`를 호출하여 그래프에 등록합니다.

> 참고 문서: LangGraph에 pre-built 되어 있는 [tools_condition](https://langchain-ai.github.io/langgraph/reference/prebuilt/#tools_condition)으로 대체할 수 있습니다.

### `add_conditional_edges` 메서드

![add_conditional_edges](./assets/langgraph-02.png)

`add_conditional_edges` 메서드는 시작 노드에서 여러 대상 노드로의 조건부 엣지를 추가합니다. 이 메서드를 사용하면 그래프 실행 중에 동적으로 다음 노드를 결정할 수 있습니다.

**매개변수**
- `source` (str): 시작 노드입니다. 이 노드를 나갈 때 조건부 엣지가 실행됩니다.
- `path` (Union[Callable, Runnable]): 다음 노드를 결정하는 호출 가능한 객체 또는 Runnable입니다. `path_map`을 지정하지 않으면 하나 이상의 노드를 반환해야 합니다. `END`를 반환하면 그래프 실행이 중지됩니다.
- `path_map` (Optional[Union[dict[Hashable, str], list[str]]]): 경로와 노드 이름 간의 매핑입니다. 생략하면 `path`가 반환하는 값이 노드 이름이어야 합니다.
- `then` (Optional[str]): `path`로 선택된 노드 실행 후 실행할 노드의 이름입니다.

**주요 기능**
1. 조건부 엣지를 그래프에 추가합니다.
2. `path_map`을 딕셔너리로 변환합니다.
3. `path` 함수의 반환 타입을 분석하여 자동으로 `path_map`을 생성할 수 있습니다.
4. 조건부 분기를 그래프에 저장합니다.

### 라우터 함수 정의 및 그래프 구성

`route_tools` 함수는 챗봇 노드의 출력을 확인하여 도구 호출 여부를 판단합니다. AI 메시지에 `tool_calls` 속성이 있고 호출 목록이 비어있지 않으면 `"tools"` 문자열을 반환하여 도구 노드로 이동하고, 그렇지 않으면 `END`를 반환하여 그래프 실행을 종료합니다.

그래프의 엣지 구성은 다음과 같습니다: `START` -> `chatbot` -> (조건부) -> `tools` 또는 `END`, 그리고 `tools` -> `chatbot`으로 돌아가는 순환 구조를 형성합니다. 이 순환 구조 덕분에 도구 실행 결과를 바탕으로 LLM이 다시 판단하여 추가 도구 호출이나 최종 응답을 생성할 수 있습니다.

아래 코드에서는 라우터 함수를 정의하고, 엣지를 추가한 후 그래프를 컴파일합니다.

In [8]:
from langgraph.graph import START, END


def route_tools(state: State):
    """도구 호출 여부에 따라 다음 노드를 결정하는 라우터 함수

    AI 메시지에 tool_calls가 있으면 'tools' 노드로,
    없으면 END로 라우팅합니다.
    """
    if messages := state.get("messages", []):
        # 가장 최근 AI 메시지 추출
        ai_message = messages[-1]
    else:
        # 입력 상태에 메시지가 없는 경우 예외 발생
        raise ValueError(f"No messages found in input state to tool_edge: {state}")

    # AI 메시지에 도구 호출이 있는 경우 'tools' 반환
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    # 도구 호출이 없는 경우 END 반환 (그래프 종료)
    return END


# 조건부 엣지 추가: chatbot 노드 출력에 따라 tools 또는 END로 라우팅
graph_builder.add_conditional_edges(
    source="chatbot",
    path=route_tools,
    # route_tools의 반환값에 따른 라우팅 매핑
    path_map={"tools": "tools", END: END},
)

# 도구 실행 후 챗봇으로 돌아가는 엣지
graph_builder.add_edge("tools", "chatbot")

# 시작 노드에서 챗봇으로 연결
graph_builder.add_edge(START, "chatbot")

# 그래프 컴파일
graph = graph_builder.compile()

### 조건부 엣지 동작 원리

조건부 엣지는 단일 노드에서 시작해야 합니다. 위 코드에서 `chatbot` 노드가 실행될 때마다 `route_tools` 함수가 호출되어 다음 노드를 결정합니다. 도구를 호출하면 `"tools"` 노드로 이동하고, 직접 응답이 가능하면 `END`로 이동하여 루프를 종료합니다.

pre-built `tools_condition` 함수와 동일한 방식으로, 도구 호출이 없을 경우 `END` 문자열을 반환하여 그래프 실행을 중지합니다. 그래프가 `END`로 전환되면 더 이상 완료할 작업이 없으므로 실행이 종료됩니다.

아래 코드에서는 컴파일된 그래프를 시각화합니다.

In [ ]:
from IPython.display import Image

# 그래프 시각화 (Excalidraw로 생성된 PNG 이미지)
Image(filename="assets/03-agent-graph.png")

---

## 그래프 실행

이제 구축된 Agent 그래프를 실행하여 웹 검색 기능을 테스트해봅니다. 봇에게 LLM의 훈련 데이터에 포함되지 않았을 수 있는 최신 정보에 대해 질문하면, Agent는 자동으로 웹 검색 도구를 호출하여 답변을 생성합니다.

`stream_graph` 함수를 사용하면 각 노드의 실행 결과를 단계별로 스트리밍하여 확인할 수 있습니다. 이를 통해 Agent가 어떤 단계를 거쳐 최종 답변을 생성하는지 명확하게 관찰할 수 있습니다.

아래 코드에서는 테디노트 YouTube 채널에 대해 검색을 요청합니다.

In [10]:
from langchain_teddynote.messages import stream_graph

# 검색 요청 입력
inputs = {"messages": "테디노트 YouTube 채널에 대해서 검색해 줘"}

# 스트리밍 실행
stream_graph(graph, inputs)


🔄 Node: chatbot 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 



🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
[{"url": "https://www.youtube.com/channel/UCt2wAAXgm87ACiQnDHQEW6Q", "title": "테디노트 TeddyNote - YouTube", "content": "데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에 관심이 많습니다 ‍♂️ ...more 데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을", "score": 0.8455478, "raw_content": null}, {"url": "https://teddylee777.github.io/lectures/", "title": "강의 - 테디노트", "content": "⑤ 서울대 **PyTorch 딥러닝 강의** 바로가기 🙌. * 본 강의를 수강하신 분들이 평균 2~3주 만에 자격증을 취득하고 있습니다. ## 💻 한 방으로 끝내는 판다스(Pandas) 데이터 분석 - 전자책 포함. ## 🌱 한 권으로 끝내는 파이썬(Python) 코딩 입문 - 전자책 포함. 파이썬 `python` 코딩 `입문자` 분들께 추천 드려요🤩. ## 🌱 깃헙 블로그(Github blog)로 차별화 된 나만의 홈페이지 만들기! 차별화된 깃헙 블로그(GitHub blog) 만들어 보고 싶으신 분들께 추천 드립니다🤩. ## 🌱 스트림릿(Streamlit)을 활용한 파이썬 웹앱 제작하기. * 스트림릿(streamlit)을 활용하여 파이썬 웹앱 대시보드를 구현해보고 싶으신 분들께 추천 드립니다🤩. * 데이터 분석/대시보드 제작 그리고 머신러닝 모델을 배포해서 서비스화 시키는 과정입니다😊. * 파이썬 기본 문법만 알고 계신다면 부담 없이 도전해 보실 수 있어요 🥰. 그 밖에도 다양한 주제로 강의 영상을 유튜브(YouTube) 채널에 업로드 하고 있어요. 유튜브 채널에서 `재생목록을 꼭 클릭`해 보세요! `주제별로 강의 영상`을 시리


🔄 Node: chatbot 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
테

디

노

...

용적인 교

육 

콘텐츠를 제

공하는

 채널입니다!

---

## 도구 호출 후 메시지 구조

도구가 호출된 후의 메시지 구조를 살펴보겠습니다. LLM이 도구를 호출하면 `AIMessage`에 `tool_calls` 속성이 포함되고, 도구 실행 결과는 `ToolMessage` 형태로 반환됩니다. 이 흐름을 이해하면 Agent의 동작 방식을 더 잘 파악할 수 있습니다.

메시지 흐름은 다음과 같습니다:
1. `HumanMessage`: 사용자의 질문
2. `AIMessage` (with `tool_calls`): LLM이 도구 호출을 결정
3. `ToolMessage`: 도구 실행 결과
4. `AIMessage`: 도구 결과를 바탕으로 생성된 최종 답변

아래 이미지는 도구 호출 후 메시지 구조를 시각적으로 보여줍니다.

![](./assets/tool-message-01.png)